# Evaluate baseline POWSM vs L2-ARCTIC LoRA — **fast version**

Speedups vs `06_eval_l2arctic.ipynb`:

1. **Test WAV'ları local SSD'ye kopyalar** (Drive FUSE'dan tek tek okumak yerine) → en büyük kazanç.
2. **`BEAM_SIZE=1` (greedy decode)** varsayılan → beam search'e göre ~3-5x hızlı. İki modele de aynı ayar uygulandığı için delta karşılaştırması adil kalır. Final rapor sayıları için `BEAM_SIZE=5` ile tekrar koşabilirsin.
3. **tqdm + canlı mean-PER + ETA** → loop'un canlı olduğunu ve ne kadar kaldığını görürsün.
4. **Resume desteği**: her utterance'ın sonucu anında JSONL'e yazılır ve periyodik olarak Drive'a yedeklenir. Oturum koparsa baştan başlamaz, kaldığı yerden devam eder.
5. `torch.inference_mode()` + pinsiz `espnet` kurulumu (pinli `espnet==202412` Colab'ın Python 3.12'sinde wheel build hatası veriyor).

Upload **both** `turkish_lora_util.py` and `l2arctic_util.py` to `MyDrive/senior/`. **Local:** paths auto-detect from the repo.

In [1]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    # NOT: espnet==202412 pini Colab'in Python 3.12'sinde build hatasi veriyor -> pinsiz kuruyoruz.
    %pip install -q espnet espnet-model-zoo "peft>=0.10" soundfile
    import espnet
    print("espnet version:", espnet.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.4/70.4 kB 6.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.0/261.0 kB 25.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.3/180.3 kB 21.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 81.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 81.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 74.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 107.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# ============================================================
# CONFIGURATION
# ============================================================
DRIVE_ROOT  = "/content/drive/MyDrive/senior"
CHUNKS_DIR  = "/content/drive/MyDrive/senior/l2arctic_chunks"   # Drive'daki klasor
LOCAL_DIR   = "/content/l2arctic_test_wavs"                      # test WAV'larin SSD kopyasi
ADAPTER_DIR = "/content/drive/MyDrive/senior/lora_checkpoints/best"
RESULTS_DIR = "/content/drive/MyDrive/senior/eval_results"       # resume dosyalari (Drive'da kalici)

MODEL_ID    = "espnet/powsm"
LANG_SYM    = "<unk>"
TASK_SYM    = "<pr>"

EVAL_SUBSET = None   # hizli kontrol icin 20 yap; tam test seti icin None
BEAM_SIZE   = 1      # 1 = greedy (hizli). Final sayilar icin 5 ile tekrar kos.
SAVE_EVERY  = 50     # kac utterance'ta bir Drive'a yedek kopyalansin
# ============================================================

In [3]:
from pathlib import Path

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    sys.path.insert(0, DRIVE_ROOT)            # util dosyalari
    sys.path.insert(0, f"{DRIVE_ROOT}/mod")   # edit_distance
else:
    _here = Path.cwd().resolve()
    FT = _here if (_here / "l2arctic_util.py").is_file() else _here.parent
    REPO_ROOT = FT.parents[1]
    sys.path.insert(0, str(FT))
    sys.path.insert(0, str(REPO_ROOT / "mod"))
    CHUNKS_DIR  = str(FT / "data" / "l2arctic_chunks")
    LOCAL_DIR   = CHUNKS_DIR                       # local'de zaten SSD'de
    ADAPTER_DIR = str(FT / "lora_checkpoints" / "best")
    RESULTS_DIR = str(FT / "eval_results")

CHUNKS_DIR  = Path(CHUNKS_DIR)
LOCAL_DIR   = Path(LOCAL_DIR)
ADAPTER_DIR = Path(ADAPTER_DIR)
RESULTS_DIR = Path(RESULTS_DIR)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("CHUNKS_DIR :", CHUNKS_DIR,  "| exists:", CHUNKS_DIR.is_dir())
print("ADAPTER_DIR:", ADAPTER_DIR, "| exists:", ADAPTER_DIR.is_dir())
print("RESULTS_DIR:", RESULTS_DIR)

Mounted at /content/drive
CHUNKS_DIR : /content/drive/MyDrive/senior/l2arctic_chunks | exists: True
ADAPTER_DIR: /content/drive/MyDrive/senior/lora_checkpoints/best | exists: True
RESULTS_DIR: /content/drive/MyDrive/senior/eval_results


In [4]:
import json

test_items = json.loads((CHUNKS_DIR / "test.json").read_text(encoding="utf-8"))
if EVAL_SUBSET:
    test_items = test_items[:EVAL_SUBSET]
print(f"{len(test_items)} test utterances will be evaluated")

900 test utterances will be evaluated


In [5]:
# Sadece test setindeki WAV'lari local SSD'ye kopyala (tum 3599 degil, ~900 dosya).
# Var olan dosyalar atlanir -> hucre tekrar calistirilabilir (resume-friendly).
import shutil
from tqdm.auto import tqdm

if IN_COLAB:
    LOCAL_DIR.mkdir(parents=True, exist_ok=True)
    copied = skipped = 0
    for item in tqdm(test_items, desc="copy test WAVs -> SSD"):
        src = CHUNKS_DIR / f"{item['id']}.wav"
        dst = LOCAL_DIR / f"{item['id']}.wav"
        if dst.is_file() and dst.stat().st_size == src.stat().st_size:
            skipped += 1
            continue
        shutil.copy2(src, dst)
        copied += 1
    print(f"copied {copied}, skipped {skipped} (already on SSD)")
    WAV_DIR = LOCAL_DIR
else:
    WAV_DIR = LOCAL_DIR
print("WAV_DIR:", WAV_DIR)

copy test WAVs -> SSD:   0%|          | 0/900 [00:00<?, ?it/s]

copied 900, skipped 0 (already on SSD)
WAV_DIR: /content/l2arctic_test_wavs


In [6]:
import os, time
import numpy as np
import soundfile as sf
import torch
from collections import defaultdict
from espnet2.bin.s2t_inference import Speech2Text

try:
    from edit_distance import edit_operations            # mod/edit_distance.py
except ImportError:
    from assessment.edit_distance import edit_operations  # mod/assessment/edit_distance.py

from l2arctic_util import patch_speech2text_lora

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# HF_TOKEN varsa indirme rate-limit'e takilmaz (Colab: sol panel -> anahtar ikonu -> HF_TOKEN ekle)
if IN_COLAB:
    try:
        from google.colab import userdata
        tok = userdata.get("HF_TOKEN")
        if tok:
            os.environ["HF_TOKEN"] = tok
            print("HF_TOKEN set (faster downloads)")
    except Exception:
        print("HF_TOKEN not set - downloads may be rate-limited (still works)")

Failed to import Flash Attention, using ESPnet default: No module named 'flash_attn'


[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package cmudict to /root/nltk_data...
[nltk_data]   Unzipping corpora/cmudict.zip.


device: cuda
GPU: NVIDIA L4
HF_TOKEN not set - downloads may be rate-limited (still works)


In [7]:
def parse_pr_tokens(raw: str) -> list:
    if "<notimestamps>" in raw:
        raw = raw.split("<notimestamps>", 1)[1]
    tokens = []
    for part in raw.strip().split("//"):
        part = part.strip().strip("/")
        if part:
            tokens.append(part)
    return tokens


def compute_per(pred: list, ref: list) -> float:
    return len(edit_operations(pred, ref)) / max(len(ref), 1)


def run_pr(s2t_model: Speech2Text, wav_path) -> list:
    speech, r = sf.read(wav_path)
    if r != 16000:
        raise ValueError(r)
    with torch.inference_mode():
        raw = s2t_model(np.asarray(speech, dtype=np.float32), text_prev="<na>")[0][0]
    return parse_pr_tokens(raw)


def eval_model(s2t_model: Speech2Text, items: list, tag: str) -> list:
    """Her utterance icin {id, l1, ref, pred, per} kaydi uretir.

    - Sonuclar aninda local JSONL'e yazilir, SAVE_EVERY'de bir Drive'a kopyalanir.
    - Ayni tag ile tekrar cagrilirsa kaldigi yerden devam eder (resume).
    """
    from tqdm.auto import tqdm
    import shutil as _sh

    local_path = Path(f"/tmp/{tag}.jsonl") if IN_COLAB else RESULTS_DIR / f"{tag}.jsonl"
    drive_path = RESULTS_DIR / f"{tag}.jsonl"

    # resume: Drive'daki yedegi local'e al
    records = {}
    if drive_path.is_file():
        if IN_COLAB and not local_path.is_file():
            _sh.copy2(drive_path, local_path)
        for line in local_path.read_text(encoding="utf-8").splitlines():
            if line.strip():
                rec = json.loads(line)
                records[rec["id"]] = rec
    if records:
        print(f"[{tag}] resume: {len(records)} utterance zaten hazir, atlaniyor")

    todo = [it for it in items if it["id"] not in records]
    f = open(local_path, "a", encoding="utf-8")
    pbar = tqdm(todo, desc=f"eval {tag}")
    t0, done_pers = time.time(), [r["per"] for r in records.values()]
    try:
        for i, item in enumerate(pbar, 1):
            ref = item["phones"]
            pred = run_pr(s2t_model, WAV_DIR / f"{item['id']}.wav")
            per = compute_per(pred, ref)
            rec = {"id": item["id"], "l1": item.get("l1", "?"), "ref": ref, "pred": pred, "per": per}
            records[item["id"]] = rec
            done_pers.append(per)
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
            f.flush()
            sec_per_utt = (time.time() - t0) / i
            pbar.set_postfix(mean_PER=f"{np.mean(done_pers):.4f}", s_per_utt=f"{sec_per_utt:.2f}")
            if IN_COLAB and (i % SAVE_EVERY == 0):
                _sh.copy2(local_path, drive_path)   # Drive yedegi
    finally:
        f.close()
        if IN_COLAB:
            _sh.copy2(local_path, drive_path)
    return [records[it["id"]] for it in items if it["id"] in records]


def summarize(recs: list):
    """(mean_per, confusion, per_l1) - kayitlardan hesaplanir, yeniden inference gerekmez."""
    pers = [r["per"] for r in recs]
    by_l1 = defaultdict(list)
    confusion = defaultdict(lambda: defaultdict(int))
    for r in recs:
        by_l1[r["l1"]].append(r["per"])
        for op in edit_operations(r["pred"], r["ref"]):
            if op[0] == "substitute":
                confusion[op[2]][op[3]] += 1
            elif op[0] == "delete":
                confusion[op[2]]["<del>"] += 1
            elif op[0] == "insert":
                confusion["<ins>"][op[2]] += 1
    per_l1 = {k: float(np.mean(v)) for k, v in sorted(by_l1.items())}
    return float(np.mean(pers)), dict(confusion), per_l1

In [8]:
print("Loading baseline POWSM...")
s2t_base = Speech2Text.from_pretrained(
    MODEL_ID, device=DEVICE, lang_sym=LANG_SYM, task_sym=TASK_SYM, beam_size=BEAM_SIZE
)
print(f"Running baseline eval (beam_size={BEAM_SIZE})...")
base_recs = eval_model(s2t_base, test_items, tag=f"baseline_beam{BEAM_SIZE}")
base_per, base_confusion, base_l1 = summarize(base_recs)
print(f"Baseline mean PER: {base_per:.4f}  ({base_per*100:.1f}%)")
print("  per-L1:", {k: round(v, 3) for k, v in base_l1.items()})

# VRAM'i LoRA modeline birak
del s2t_base
if DEVICE == "cuda":
    torch.cuda.empty_cache()

Loading baseline POWSM...


Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Running baseline eval (beam_size=1)...
[baseline_beam1] resume: 550 utterance zaten hazir, atlaniyor


eval baseline_beam1:   0%|          | 0/350 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/espnet2/s2t/espnet_model.py:338: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(False):


Baseline mean PER: 0.3876  (38.8%)
  per-L1: {'arabic': 0.362, 'hindi': 0.331, 'korean': 0.404, 'mandarin': 0.387, 'spanish': 0.382, 'vietnamese': 0.459}


In [11]:
%pip uninstall -q -y torchao

In [12]:
if not ADAPTER_DIR.is_dir():
    print(f"Adapter not found at {ADAPTER_DIR} - skipping LoRA eval")
else:
    print("Loading LoRA model...")
    s2t_lora = Speech2Text.from_pretrained(
        MODEL_ID, device=DEVICE, lang_sym=LANG_SYM, task_sym=TASK_SYM, beam_size=BEAM_SIZE
    )
    patch_speech2text_lora(s2t_lora, ADAPTER_DIR)
    print(f"Running LoRA eval (beam_size={BEAM_SIZE})...")
    lora_recs = eval_model(s2t_lora, test_items, tag=f"lora_beam{BEAM_SIZE}")
    lora_per, lora_confusion, lora_l1 = summarize(lora_recs)
    print(f"LoRA    mean PER: {lora_per:.4f}  ({lora_per*100:.1f}%)")
    print(f"Delta           : {(lora_per - base_per)*100:+.1f} pp")
    print("  per-L1:", {k: round(v, 3) for k, v in lora_l1.items()})

Loading LoRA model...


Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Running LoRA eval (beam_size=1)...


eval lora_beam1:   0%|          | 0/900 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/espnet2/s2t/espnet_model.py:338: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(False):


LoRA    mean PER: 0.1924  (19.2%)
Delta           : -19.5 pp
  per-L1: {'arabic': 0.167, 'hindi': 0.181, 'korean': 0.208, 'mandarin': 0.207, 'spanish': 0.174, 'vietnamese': 0.218}


In [13]:
# L2 substitution targets + collapsed vowels
L2_PHONES = ["θ", "ð", "w", "ŋ", "ɹ", "v", "æ", "ɛ", "ɪ", "ʊ", "ʌ", "ə"]
if ADAPTER_DIR.is_dir():
    print(f"{'phone':<6}{'base_err':>10}{'lora_err':>10}")
    print("-" * 26)
    for ph in L2_PHONES:
        b = sum(base_confusion.get(ph, {}).values())
        l = sum(lora_confusion.get(ph, {}).values())
        mark = " v" if l < b else (" ^" if l > b else "")
        print(f"{ph:<6}{b:>10}{l:>10}{mark}")

phone   base_err  lora_err
--------------------------
θ             57        47 v
ð             51       159 ^
w            641       167 v
ŋ             50        44 v
ɹ            705       161 v
v             69        95 ^
æ            394       148 v
ɛ            364       189 v
ɪ            889       352 v
ʊ             60        72 ^
ʌ            302       200 v
ə           1100       447 v


In [14]:
# Per-L1 karsilastirma tablosu
if ADAPTER_DIR.is_dir():
    print(f"{'L1':<12}{'base':>8}{'lora':>8}{'delta':>9}")
    print("-" * 37)
    for l1 in sorted(set(base_l1) | set(lora_l1)):
        b, l = base_l1.get(l1, float("nan")), lora_l1.get(l1, float("nan"))
        print(f"{l1:<12}{b:>8.3f}{l:>8.3f}{(l-b)*100:>+8.1f}pp")

L1              base    lora    delta
-------------------------------------
arabic         0.362   0.167   -19.4pp
hindi          0.331   0.181   -15.0pp
korean         0.404   0.208   -19.6pp
mandarin       0.387   0.207   -18.1pp
spanish        0.382   0.174   -20.9pp
vietnamese     0.459   0.218   -24.1pp
